# nb7 — Phase 2 · Bước 2: Run 3 (noise v2 hiệu chỉnh + gate Telex/VNI) + post-processing

Notebook thứ bảy (`DESIGN.md` §12, quyết định 24/09): **Run 3 = đúng công thức Run 2** (LoRA r=16 α=32 dropout 0,05 targets q/v/k/out_proj · 6 epochs · batch 8×accum4 · lr 2e-4 · fp16 · seed 42 · augmentation 30% sạch + 30% nhiễu) — **chỉ đổi thành phần noise**:

- **Noise v2 hiệu chỉnh**: nâng `P_EMPIRICAL` (quét {0,5 · 0,65 · 0,8}) + giới hạn acceptance rule keyboard + (nếu gate pass) slot Telex/VNI share nhỏ — hiệu chuẩn **non-word rate sinh về [23%, 25%]** (đối chiếu thật train ~23,4%; nb2 v1 lệch lên 51,6% — `REPORT.md` §8.6).
- **Gate Telex** (heuristic tất định theo plan WP2 §2): pair 1-token train có char-edit-distance(error, correction) ≥ 2 **HOẶC** error chứa ký tự ASCII lặp (vd `dd`, `ww`, `oo` sau khi bỏ dấu) → **≥ 5%** thì bật rule `telex_grammar`; < 5% → log bỏ (lỗ hổng telex ghi nhận là giới hạn đồ án).
- **Post-processing**: re-tune **cùng rule-shape nb6** trên **val Run 3** (mining lại whitelist/blacklist từ val Run 3, không tái dùng Run 2) → frozen lên test Run 3.

**Kaggle GPU (T4)** — chỉ **1 run train** trong plan (tiết kiệm quota); guard `transformers<5` như nb5 (bug v5 với tokenizer sentencepiece). Mục tiêu non-word [23, 25]% chốt **trước** khi nhìn test (không test-peeking; stratified chỉ report).

**Input**: nb0 (`vsec_train.jsonl`, `vsec_val.jsonl`) · nb1 (`test_aligned.jsonl`) · nb2 (`syllable_table.json`, `noise_model.json`) · nb6 (`postprocess_config.json`) · optional nb3 (`predictions_{val,test}_run2.jsonl` — so trực diện Run 2).

**Output**: `lora_adapter_run3/` · `predictions_{val,test}_run3.jsonl` · `eval_report_run3.json` · `noise_model_v2.json` · `postprocess_run3_report.json`.

**Checklist chống leakage (`DESIGN.md` §9)**: augmentation chỉ `split == 'train'` (assert mọi record) · confusion/noise model từ nb2 (nguồn train) · bảng âm tiết nguồn công khai độc lập · post-processing tune trên val Run 3 · seed 42 cố định.

## Nội dung
0. Cấu hình + guard transformers + tự dò input
1. Cell hàm dùng chung (copy nguyên vẹn từ nb5) + `evaluate_predictions`
2. Dữ liệu + noise model v1 + đo real non-word rate train
3. Gate Telex trên pair train
4. Noise v2 builder + calibration grid + xuất `noise_model_v2.json`
5. Build augmentation Run 3 + train (công thức nb3)
6. Inference + eval 3 chỉ số val/test — so trực diện Run 2 (kèm stratified)
7. Post-processing re-tune trên val Run 3 → frozen test Run 3
8. Xuất file

In [1]:
!pip uninstall -y torchao==0.10.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 0. Cấu hình & guard môi trường

Mọi tham số gom một chỗ (`DESIGN.md` §7). Công thức train giữ nguyên nb3; nhóm tham số mới chỉ dành cho noise v2 + gate telex + re-tune post-processing (`MIN_VAL_GAIN` cùng ngưỡng 0,3 F1 với nb6).

In [2]:
import os
import json
import random
import datetime
import unicodedata
from pathlib import Path
import collections

# Cài đặt thư viện cần thiết nếu chạy trên Kaggle (giống nb3/nb5)
try:
    import peft
    import sentencepiece
except ImportError:
    print('Cài đặt peft và sentencepiece...')
    os.system('pip install -q peft sentencepiece')
    os.system('pip uninstall -y torchao')

# Guard version: transformers v5 bug convert tokenizer sentencepiece (ViT5/T5-style) → KeyError: 0
# (nb5 đã chạy thành công với cơ chế này ngày 24/09 — copy nguyên)
import transformers
# from packaging.version import Version
# if Version(transformers.__version__).major >= 5:
#     print(f'transformers {transformers.__version__} (v5) — downgrade về <5...')
#     os.system('pip install -q "transformers<5"')
#     raise SystemExit(
#         'Đã downgrade transformers về <5. BÂY GIỜ: Restart Kernel '
#         '(Run → Restart & Clear Outputs) rồi Run All lại từ đầu — guard sẽ pass và chạy bình thường.'
#     )

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed
)
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
set_seed(SEED)

# --- Công thức Run 2 (GIỮ NGUYÊN — quyết định 24/09) ---
MODEL_NAME = 'vinai/bartpho-syllable'
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]
BATCH_SIZE = 8
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
EPOCHS = 6
FP16 = torch.cuda.is_available()
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256
AUG_CLEAN_RATIO = 0.3
AUG_NOISE_RATIO = 0.3
EVAL_BEAM_SIZE = 1

# --- Noise v2 (mới) ---
P_EMPIRICAL_GRID = [0.5, 0.65, 0.8]      # quét xác suất dùng confusion thực nghiệm
KEYBOARD_STRICT_GRID = [False, True]     # True: rule keyboard chỉ nhận candidate ∈ bảng âm tiết ∨ ERROR_VOCAB
NONWORD_TARGET_LO, NONWORD_TARGET_HI = 0.23, 0.25  # mục tiêu non-word rate sinh (thật train ~23,4%)
CALIB_N = 2000                            # số câu sinh cho calibration
TELEX_GATE_MIN = 0.05                     # gate: >= 5% pair train kiểu telex → bật rule
TELEX_SHARE_GRID = [0.0, 0.05, 0.10]      # slot telex share nhỏ (chỉ dùng khi gate pass)

# --- Post-processing re-tune (cùng rule-shape nb6) ---
MIN_VAL_GAIN = 0.003
PHRASE_MIN_COUNT = 2

REQUIRED_FILES = [
    'vsec_train.jsonl', 'vsec_val.jsonl',       # nb0
    'test_aligned.jsonl',                        # nb1
    'syllable_table.json', 'noise_model.json',   # nb2
    'postprocess_config.json',                   # nb6
]
OPTIONAL_FILES = ['predictions_val_run2.jsonl', 'predictions_test_run2.jsonl']  # nb3 — so trực diện


def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
    local = Path('./out')
    if local.is_dir():
        candidates += sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
    found = {}
    for p in candidates:
        if (p.name in REQUIRED_FILES or p.name in OPTIONAL_FILES) and p.name not in found:
            found[p.name] = p
    missing = [req for req in REQUIRED_FILES if req not in found]
    if missing:
        raise FileNotFoundError(
            'Thiếu input bắt buộc: ' + ', '.join(missing) +
            ' | nb7 cần Add Input: output nb0 (vsec_train/val), nb1 (test_aligned), '
            'nb2 (syllable_table + noise_model), nb6 (postprocess_config). '
            'Optional nb3 (predictions_{val,test}_run2.jsonl) cho so trực diện Run 2.'
        )
    return found


INPUT_FILES = find_required_inputs()
OPTIONAL_PRESENT = {k: INPUT_FILES[k] for k in OPTIONAL_FILES if k in INPUT_FILES}

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

print('=== TỰ DÒ INPUT HOÀN TẤT ===')
for k in REQUIRED_FILES:
    print(f'  {k:28s}: {INPUT_FILES[k]}')
print('Optional (nb3, so trực diện):', OPTIONAL_PRESENT if OPTIONAL_PRESENT else 'không có — in số tham chiếu REPORT.md')
print(f'Device: {"CUDA " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (⚠️ train Run 3 cần GPU T4)"} | FP16={FP16}')

=== TỰ DÒ INPUT HOÀN TẤT ===
  vsec_train.jsonl            : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/vsec_train.jsonl
  vsec_val.jsonl              : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/vsec_val.jsonl
  test_aligned.jsonl          : /kaggle/input/notebooks/cquangnguynl/nb1-align-annotate/test_aligned.jsonl
  syllable_table.json         : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/syllable_table.json
  noise_model.json            : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/noise_model.json
  postprocess_config.json     : /kaggle/input/notebooks/cquangnguynl/nb6-post-processing/postprocess_config.json
Optional (nb3, so trực diện): {'predictions_val_run2.jsonl': PosixPath('/kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_val_run2.jsonl'), 'predictions_test_run2.jsonl': PosixPath('/kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_test_run2.jsonl')}
Device: CUDA Tesla T4 | FP16=True


## 1. Cell hàm dùng chung — copy NGUYÊN VẸT từ nb5 (`align-v1`) + `evaluate_predictions`

Như nb3/nb5/nb6: shared cell của nb5 là bản hợp nhất đầy đủ (có `build_pseudo_annotation` + `SUSPECT_EDIT_RATIO`); hàm eval giữ nguyên sanity case nhân tạo.

In [3]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

SUSPECT_EDIT_RATIO = 0.3  # giữ nguyên giá trị nb1/nb2 — cell hàm dùng chung copy nguyên vẹn cần nó

def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (các notebook sau phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

SHARED_CELLS_VERSION: align-v1


### Hàm đánh giá chuẩn 3 lớp chỉ số — copy nguyên vẹn từ nb5 (giống nb3 §1)

`evaluate_predictions(records, predictions, syll_set)` chỉ yêu cầu record có khóa `text` + `corrected_text`; sanity case nhân tạo giữ nguyên.

In [4]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định (copy nguyên vẹn từ nb3)
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')

Sanity check hàm evaluate_predictions PASS 100%!


## 2. Dữ liệu + noise model v1 + real non-word rate train

Assert mọi record train `split == 'train'`; tái tạo rule generator từ `noise_model.json` của nb2 (copy nb3 §2); đo **real non-word rate trên train** (cùng phép đo nb2 §3a, pair 1-token) làm mốc hiệu chuẩn [23%, 25%].

In [5]:
train_records = load_jsonl(INPUT_FILES['vsec_train.jsonl'])
val_records = load_jsonl(INPUT_FILES['vsec_val.jsonl'])
test_records = load_jsonl(INPUT_FILES['test_aligned.jsonl'])

for r in train_records:
    assert r.get('split') == 'train', f'Data leakage alert: record {r.get("row_id")} không phải split=train'
print(f'Train: {len(train_records)} (assert split=train PASS) · Val: {len(val_records)} · Test: {len(test_records)}')

SYLL_TABLE = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
SYLL_SET = set(SYLL_TABLE['entries'])
print(f'Bảng âm tiết: {len(SYLL_SET)} entries')

# --- Tái tạo noise model v1 từ noise_model.json (copy nb3 §2) ---
noise_model_v1 = json.loads(Path(INPUT_FILES['noise_model.json']).read_text(encoding='utf-8'))
MAX_ERRORS_PER_SENT = noise_model_v1['config']['max_errors_per_sent']
QWERTY_ADJ = noise_model_v1['config']['qwerty_adj']
VOWEL_MAP = noise_model_v1['config']['vowel_map']
REGIONAL_SWAPS = [tuple(x) for x in noise_model_v1['config']['regional_swaps']]
TONE_MARKS = [chr(int(x, 16)) for x in noise_model_v1['config']['tone_marks']]
CONFUSION = noise_model_v1['confusion']
ERROR_COUNT_DIST = {int(k): v for k, v in noise_model_v1['error_count_dist'].items()}
ERROR_VOCAB = {e for errs in CONFUSION.values() for e in errs}


def char_lev_at_most_1(a, b):  # copy nb2/nb3
    if abs(len(a) - len(b)) > 1:
        return False
    if len(a) < len(b):
        a, b = b, a
    i = j = diff = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            i += 1
            j += 1
        else:
            diff += 1
            if diff > 1:
                return False
            if len(a) == len(b):
                i += 1
                j += 1
            else:
                i += 1
    if i < len(a) or j < len(b):
        diff += 1
    return diff <= 1


def rule_candidates(syl):  # copy nb3 §2 (regional/tone/vowel/keyboard)
    cands = {}
    for src, dst in REGIONAL_SWAPS:
        if syl.startswith(src):
            rest = syl[len(src):]
            if rest and rest[0] in set('aăâeêioôơuưy'):
                cands[dst + rest] = 'regional'
    d = unicodedata.normalize('NFD', syl)
    for mark in TONE_MARKS:
        if mark in d:
            for m in TONE_MARKS:
                if m != mark:
                    cands[unicodedata.normalize('NFC', d.replace(mark, m))] = 'tone'
    for i, ch in enumerate(syl):
        for alt in VOWEL_MAP.get(ch, ''):
            if alt != ch:
                cands[syl[:i] + alt + syl[i+1:]] = 'vowel'
    for i, ch in enumerate(syl):
        if ch.isascii() and ch.isalpha():
            for rep in QWERTY_ADJ.get(ch.lower(), ''):
                cands[syl[:i] + rep + syl[i+1:]] = 'keyboard'
            if len(syl) > 1:
                cands[syl[:i] + syl[i+1:]] = 'keyboard'
            cands[syl[:i] + ch + syl[i:]] = 'keyboard'
    return cands


def accepted_rule_candidates(low_tok, syll_set, keyboard_strict=False):
    """nb3 §2 + nhánh v2: keyboard_strict=True → candidate keyboard phải ∈ bảng âm tiết ∨ ERROR_VOCAB (bỏ lev<=1)."""
    cands = {}
    for c, src in rule_candidates(low_tok).items():
        if not c or c == low_tok:
            continue
        if src == 'keyboard' and keyboard_strict:
            if c in syll_set or c in ERROR_VOCAB:
                cands[c] = 'rule:keyboard'
        elif c in syll_set or c in ERROR_VOCAB or char_lev_at_most_1(c, low_tok):
            cands[c] = 'rule:' + src
    return cands


def is_nonword(tok, syll_set):
    return nfc_normalize(tok).lower() not in syll_set


def measure_error_side(records, syll_set):  # copy nb2 §3a (đếm pair 1-token)
    res = collections.Counter()
    for rec in records:
        for pair in rec.get('correction_pairs') or []:
            toks = [t for t in canon_tokenize(pair.get('error', '')) if not is_punct_token(t)]
            if not toks:
                res['skipped_empty'] += 1
            elif len(toks) == 1 and not is_word_token(toks[0]):
                res['skipped_digit'] += 1
            elif len(toks) == 1:
                res['nonword' if is_nonword(toks[0], syll_set) else 'realword'] += 1
            else:
                res['structural'] += 1
    return res


def nonword_rate(m):
    sub = m['nonword'] + m['realword']
    return m['nonword'] / sub if sub else 0.0


train_err = measure_error_side(train_records, SYLL_SET)
REAL_NONWORD_RATE = nonword_rate(train_err)
print(f"Real non-word rate train: {REAL_NONWORD_RATE:.1%} "
      f"({train_err['nonword']}/{train_err['nonword'] + train_err['realword']} pair 1-token) — "
      f"mục tiêu calibration [{NONWORD_TARGET_LO:.0%}, {NONWORD_TARGET_HI:.0%}] (nb2 v1 sinh lệch 51,6%)")

Train: 8343 (assert split=train PASS) · Val: 927 · Test: 5983
Bảng âm tiết: 7884 entries
Real non-word rate train: 23.4% (2344/10032 pair 1-token) — mục tiêu calibration [23%, 25%] (nb2 v1 sinh lệch 51,6%)


## 3. Gate Telex trên pair train

Heuristic tất định (plan WP2 §2, đo **chỉ trên train** — không test-peeking): pair 1-token (error, correction) là telex-like nếu char-edit-distance ≥ 2 HOẶC error (đã bỏ dấu) chứa ký tự ASCII lặp. ≥ 5% → bật rule; < 5% → bỏ, ghi nhận giới hạn.

In [6]:
def ascii_fold(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))


def char_levenshtein(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(prev + (0 if a[i - 1] == b[j - 1] else 1), dp[j] + 1, dp[j - 1] + 1)
            prev = cur
    return dp[m]


def telex_like(err_low, corr_low):
    """Heuristic plan WP2 §2: telex-like nếu char-edit-distance(error, correction) >= 2
    HOẶC error (đã bỏ dấu) chứa ký tự ASCII lặp (vd dd, ww, oo)."""
    if not err_low or not corr_low or err_low == corr_low:
        return False
    if char_levenshtein(err_low, corr_low) >= 2:
        return True
    e = ascii_fold(err_low)
    return any(e[i] == e[i + 1] and e[i].isascii() and e[i].isalpha() for i in range(len(e) - 1))


telex_pairs = 0
kept_pairs = 0
telex_examples = []
for rec in train_records:
    for pair in rec.get('correction_pairs') or []:
        e_toks = [t for t in canon_tokenize(pair.get('error', '')) if is_word_token(t)]
        c_toks = [t for t in canon_tokenize(pair.get('correction', '')) if is_word_token(t)]
        if len(e_toks) != 1 or len(c_toks) != 1:
            continue
        kept_pairs += 1
        if telex_like(e_toks[0].lower(), c_toks[0].lower()):
            telex_pairs += 1
            if len(telex_examples) < 15:
                telex_examples.append((e_toks[0], c_toks[0]))

TELEX_RATE = telex_pairs / kept_pairs if kept_pairs else 0.0
TELEX_RULE_ENABLED = TELEX_RATE >= TELEX_GATE_MIN
print(f'Gate Telex: {telex_pairs}/{kept_pairs} pair train telex-like = {TELEX_RATE:.1%} '
      f'(ngưỡng >= {TELEX_GATE_MIN:.0%})')
print('  Mẫu pair telex-like:', telex_examples)
if TELEX_RULE_ENABLED:
    print('  → Gate PASS: bật rule telex_grammar (slot share nhỏ trong noise v2).')
else:
    TELEX_SHARE_GRID = [0.0]
    print('  → Gate FAIL: bỏ rule telex (lỗ hổng telex = giới hạn đồ án đã ghi nhận DESIGN.md §12).')

Gate Telex: 1395/9533 pair train telex-like = 14.6% (ngưỡng >= 5%)
  Mẫu pair telex-like: [('a', 'của'), ('khoá', 'khóa'), ('Truyện', 'Chuyện'), ('nghĩaa', 'nghĩa'), ('scar', 'cả'), ('xủa', 'xử'), ('giàng', 'dành'), ('thoả', 'thỏa'), ('đuwowcj', 'được'), ('tuỳ', 'tùy'), ('hoà', 'hòa'), ('sát', 'xác'), ('doạ', 'dọa'), ('hoàn', 'hòa'), ('thoả', 'thỏa')]
  → Gate PASS: bật rule telex_grammar (slot share nhỏ trong noise v2).


## 4. Noise v2 builder + calibration (seeded, tất định)

Generator giữ nguyên logic v1 (confusion thực nghiệm + rule regional/tone/vowel/keyboard) cộng 2 biến:
- `keyboard_strict = True`: rule keyboard chỉ nhận candidate ∈ bảng âm tiết ∨ ERROR_VOCAB (bỏ nhánh edit-distance ≤ 1 — nguồn non-word rác lớn của v1).
- Slot `telex_share` (chỉ khi gate pass): với xác suất share, vị trí eligible bị hỏng kiểu **telex/VNI tất định** (bảng chuyển tự: đ→`dd`/`d9`, â→`aa`/`a6`, ă→`aw`/`a8`, ê→`ee`/`e6`, ô→`oo`/`o6`, ơ→`ow`/`o7`, ư→`uw`/`u7` + key thanh `s f r x j` / `1 2 3 4 5`).

**Calibration**: quét grid {`P_EMPIRICAL`} × {`keyboard_strict`} × {`telex_share`} trên `CALIB_N` câu sinh (cùng mẫu câu seeded giữa các tổ hợp); đo non-word rate sinh (cùng phép nb2 §6: non-word edits / tổng edits) → **chọn bộ trọng số đưa rate về [23%, 25%]** (ưu tiên trong vùng, gần midpoint 24%); không tổ hợp nào đạt → nhận mức gần nhất + log, không ép cực đoan. Assert NFC mọi đầu ra + invariant vị trí + deterministic cùng seed.

In [7]:
_TONE_KEY_TELEX = {0x0301: 's', 0x0300: 'f', 0x0309: 'r', 0x0303: 'x', 0x0323: 'j'}
_TONE_KEY_VNI = {0x0301: '1', 0x0300: '2', 0x0309: '3', 0x0303: '4', 0x0323: '5'}
_CIRC, _BREVE, _HORN = 0x0302, 0x0306, 0x031B
_CIRC_KEY_TELEX = {'a': 'a', 'e': 'e', 'o': 'o'}
_CIRC_KEY_VNI = {'a': '6', 'e': '6', 'o': '6'}
_BREVE_KEY = {'telex': 'w', 'vni': '8'}
_HORN_KEY = {'telex': 'w', 'vni': '7'}


def transliterate_style(syl, style):
    """Chuyển tự tất định 1 âm tiết NFC sang chuỗi telex/VNI (đầu ra ASCII → NFC-safe)."""
    tone_key = _TONE_KEY_TELEX if style == 'telex' else _TONE_KEY_VNI
    circ_key = _CIRC_KEY_TELEX if style == 'telex' else _CIRC_KEY_VNI
    out = []
    for ch in syl:
        if ch == 'đ':
            out.append('dd' if style == 'telex' else 'd9')
            continue
        d = unicodedata.normalize('NFD', ch)
        base, suffix = d[0], ''
        for mk in d[1:]:
            o = ord(mk)
            if o == _CIRC:
                suffix += circ_key.get(base, '')
            elif o == _BREVE:
                suffix += _BREVE_KEY[style]
            elif o == _HORN:
                suffix += _HORN_KEY[style]
            else:
                suffix += tone_key.get(o, '')
        out.append(base + suffix)
    return ''.join(out)


def telex_candidates(low_syl):
    cands = []
    for style in ('telex', 'vni'):
        t = transliterate_style(low_syl, style)
        if t != low_syl and t not in cands:
            cands.append(t)
    return cands


# Sanity chuyển tự
assert transliterate_style('đã', 'telex') == 'ddax', transliterate_style('đã', 'telex')
assert transliterate_style('đã', 'vni') == 'd9a4', transliterate_style('đã', 'vni')
assert transliterate_style('trường', 'telex') == 'truwowfng', transliterate_style('trường', 'telex')  # ư→uw, ơ→ow
assert telex_candidates('ơ')
assert all(unicodedata.is_normalized('NFC', c) for c in telex_candidates('được'))
print('Sanity chuyển tự telex/VNI PASS:',
      transliterate_style('được', 'telex'), '/', transliterate_style('được', 'vni'))


def generate_noisy_v2(clean_text, rng, syll_set, p_empirical, keyboard_strict,
                      telex_share, telex_enabled):
    """Generator v1 (nb3 §2) + v2: keyboard_strict + slot telex. Trả về dict {clean_text, noisy_text, edits} hoặc None.
    telex_share = 0 hoặc telex_enabled = False → hành vi trùng v1 (cùng rng stream)."""
    toks = canon_tokenize(clean_text)
    cand_cache = {}
    eligible = []
    for i, tok in enumerate(toks):
        if not is_word_token(tok):
            continue
        low = nfc_normalize(tok).lower()
        opts = {}
        for e in CONFUSION.get(low, {}):
            if e and e != low:
                opts[e] = 'empirical'
        for c, src in accepted_rule_candidates(low, syll_set, keyboard_strict).items():
            opts.setdefault(c, src)
        t_cands = telex_candidates(low) if (telex_enabled and telex_share > 0) else []
        if opts or t_cands:
            eligible.append(i)
            cand_cache[i] = (low, opts, t_cands)
    if not eligible:
        return None
    k = min(rng.choices(list(ERROR_COUNT_DIST), weights=list(ERROR_COUNT_DIST.values()), k=1)[0], len(eligible))
    out = list(toks)
    edits = []
    for i in sorted(rng.sample(eligible, k)):
        low, opts, t_cands = cand_cache[i]
        pick = None
        if t_cands and rng.random() < telex_share:
            pick = rng.choice(t_cands)
            source = 'telex'
        else:
            emp = {c: CONFUSION[low][c] for c, s in opts.items() if s == 'empirical'}
            if emp and rng.random() < p_empirical:
                pick = rng.choices(list(emp), weights=list(emp.values()), k=1)[0]
                source = 'empirical'
            else:
                rules = [c for c, s in opts.items() if s.startswith('rule:')]
                if rules:
                    pick = rng.choice(rules)
                    source = opts[pick]
                elif emp:
                    pick = rng.choices(list(emp), weights=list(emp.values()), k=1)[0]
                    source = 'empirical'
        if pick is None:
            continue  # vị trí chỉ có candidate telex nhưng roll hụt → bỏ vị trí (không ép sửa)
        cand = pick[:1].upper() + pick[1:] if toks[i][:1].isupper() else pick
        out[i] = cand
        edits.append({'position': i, 'original': toks[i], 'error': cand, 'source': source})
    assert 0 <= len(edits) <= k
    assert all(e['error'] for e in edits), clean_text[:60]
    noisy_text = ' '.join(out)
    assert unicodedata.is_normalized('NFC', noisy_text), clean_text
    return {'clean_text': ' '.join(toks), 'noisy_text': noisy_text, 'edits': edits}

Sanity chuyển tự telex/VNI PASS: dduwowjc / d9u7o75c


In [8]:
# Mẫu câu seeded CHUNG cho mọi tổ hợp (so sánh công bằng giữa các combo grid)
_calib_rng = random.Random(SEED)
CALIB_SENTS = []
for text in _calib_rng.sample([r['corrected_text'] for r in train_records],
                              min(CALIB_N * 2, len(train_records))):
    if len(CALIB_SENTS) >= CALIB_N:
        break
    CALIB_SENTS.append(text)
print(f'Calibration trên {len(CALIB_SENTS)} câu (cùng mẫu seeded cho mọi tổ hợp) · real train = {REAL_NONWORD_RATE:.1%}')

calib_rows = []
for p_emp in P_EMPIRICAL_GRID:
    for kb_strict in KEYBOARD_STRICT_GRID:
        for t_share in TELEX_SHARE_GRID:
            rng = random.Random(SEED + 1)
            edits_all = []
            n_generated = 0
            for text in CALIB_SENTS:
                g = generate_noisy_v2(text, rng, SYLL_SET, p_emp, kb_strict, t_share, TELEX_RULE_ENABLED)
                if g is None:
                    continue
                n_generated += 1
                changed = [i for i, (a, b) in
                           enumerate(zip(g['clean_text'].split(), g['noisy_text'].split())) if a != b]
                assert changed == [e['position'] for e in g['edits']], g['clean_text'][:60]
                edits_all.extend(g['edits'])
            rate = (sum(1 for e in edits_all if is_nonword(e['error'], SYLL_SET)) / len(edits_all)) if edits_all else 0.0
            src_dist = collections.Counter(e['source'] for e in edits_all)
            calib_rows.append({'p_empirical': p_emp, 'keyboard_strict': kb_strict, 'telex_share': t_share,
                               'nonword_rate': rate, 'n_generated': n_generated, 'n_edits': len(edits_all),
                               'source_breakdown': dict(src_dist.most_common())})
            print(f"  p_emp={p_emp:.2f} kb_strict={str(kb_strict):<5s} telex={t_share:.2f} "
                  f"→ non-word {rate:.1%} ({len(edits_all)} edits · {dict(src_dist.most_common())})")

# Deterministic same-seed check
_ga = generate_noisy_v2(CALIB_SENTS[0], random.Random(SEED + 1), SYLL_SET, 0.65, True, 0.05, TELEX_RULE_ENABLED)
_gb = generate_noisy_v2(CALIB_SENTS[0], random.Random(SEED + 1), SYLL_SET, 0.65, True, 0.05, TELEX_RULE_ENABLED)
assert _ga == _gb, 'Generator không deterministic với cùng seed!'

# Chọn bộ trọng số: ưu tiên trong [23%, 25%], gần midpoint 24%; không đạt → gần nhất + log
_target = (NONWORD_TARGET_LO + NONWORD_TARGET_HI) / 2
_in_range = [r for r in calib_rows if NONWORD_TARGET_LO <= r['nonword_rate'] <= NONWORD_TARGET_HI]
_order = {(r['p_empirical'], r['keyboard_strict'], r['telex_share']): i for i, r in enumerate(calib_rows)}
_pool = _in_range if _in_range else calib_rows
best_calib = min(_pool, key=lambda r: (abs(r['nonword_rate'] - _target),
                                       _order[(r['p_empirical'], r['keyboard_strict'], r['telex_share'])]))
CALIB_OK = bool(_in_range)
V2_PARAMS = {'p_empirical': best_calib['p_empirical'], 'keyboard_strict': best_calib['keyboard_strict'],
             'telex_share': best_calib['telex_share'] if TELEX_RULE_ENABLED else 0.0,
             'telex_enabled': TELEX_RULE_ENABLED}
print(f">> Chon noise v2: {V2_PARAMS} → non-word {best_calib['nonword_rate']:.1%} "
      f"({'trong vung muc tieu' if CALIB_OK else 'GAN NHAT — khong ep cuc doan, log muc gan nhat'}) "
      f"vs that train {REAL_NONWORD_RATE:.1%}")

noise_model_v2 = {
    'created': RUN_STAMP,
    'notebook': 'nb7_run3_train_eval',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'seed': SEED,
    'base': 'noise_model.json (nb2, nguon train) — tai tao rule nhu nb3 §2',
    'gate_telex': {'rate': TELEX_RATE, 'threshold': TELEX_GATE_MIN, 'enabled': TELEX_RULE_ENABLED,
                   'n_pairs_kept': kept_pairs, 'n_pairs_telex_like': telex_pairs, 'examples': telex_examples},
    'calibration': {'n_sentences': len(CALIB_SENTS),
                    'target_nonword_rate': [NONWORD_TARGET_LO, NONWORD_TARGET_HI],
                    'real_train_nonword_rate': REAL_NONWORD_RATE, 'in_range': CALIB_OK,
                    'grid': calib_rows,
                    'chosen': {**V2_PARAMS, 'nonword_rate': best_calib['nonword_rate']}},
    'v2_config': {'p_empirical': V2_PARAMS['p_empirical'], 'keyboard_strict': V2_PARAMS['keyboard_strict'],
                  'telex_share': V2_PARAMS['telex_share'], 'telex_enabled': V2_PARAMS['telex_enabled'],
                  'max_errors_per_sent': MAX_ERRORS_PER_SENT},
    'v1_config_ref': noise_model_v1['config'],
    'error_count_dist': dict(sorted(ERROR_COUNT_DIST.items())),
    'confusion': CONFUSION,
}
with open(OUTPUT_DIR / 'noise_model_v2.json', 'w', encoding='utf-8') as f:
    json.dump(noise_model_v2, f, ensure_ascii=False)
print('Đã ghi noise_model_v2.json vào', OUTPUT_DIR)

Calibration trên 2000 câu (cùng mẫu seeded cho mọi tổ hợp) · real train = 23.4%
  p_emp=0.50 kb_strict=False telex=0.00 → non-word 52.5% (2376 edits · {'empirical': 1092, 'rule:keyboard': 1055, 'rule:tone': 188, 'rule:vowel': 34, 'rule:regional': 7})
  p_emp=0.50 kb_strict=False telex=0.05 → non-word 53.7% (2386 edits · {'empirical': 1044, 'rule:keyboard': 1023, 'rule:tone': 182, 'telex': 92, 'rule:vowel': 38, 'rule:regional': 7})
  p_emp=0.50 kb_strict=False telex=0.10 → non-word 55.7% (2403 edits · {'empirical': 1028, 'rule:keyboard': 955, 'telex': 198, 'rule:tone': 186, 'rule:vowel': 33, 'rule:regional': 3})
  p_emp=0.50 kb_strict=True  telex=0.00 → non-word 30.2% (2387 edits · {'empirical': 1103, 'rule:keyboard': 598, 'rule:tone': 521, 'rule:vowel': 140, 'rule:regional': 25})
  p_emp=0.50 kb_strict=True  telex=0.05 → non-word 32.2% (2376 edits · {'empirical': 1093, 'rule:keyboard': 528, 'rule:tone': 474, 'rule:vowel': 166, 'telex': 83, 'rule:regional': 32})
  p_emp=0.50 kb_strict=T

## 5. Augmentation Run 3 + train (công thức nb3 giữ nguyên)

30% câu sạch (identity từ `corrected_text` train) + 30% câu nhiễu **v2** (bộ trọng số vừa chọn). Assert: nguồn augmentation chỉ từ train + NFC + invariant vị trí từng câu sinh. Hyperparameters/tokenizer/LoRA/trainer copy nguyên nb3 §3–§6.

In [9]:
rng = random.Random(SEED)
n_clean = int(len(train_records) * AUG_CLEAN_RATIO)
n_noise = int(len(train_records) * AUG_NOISE_RATIO)

clean_samples = rng.sample(train_records, n_clean)
clean_aug = [{'text': r['corrected_text'], 'corrected_text': r['corrected_text']} for r in clean_samples]

noise_samples = rng.sample(train_records, min(n_noise * 2, len(train_records)))
noisy_aug = []
n_skipped_no_candidate = 0
for r in noise_samples:
    g = generate_noisy_v2(r['corrected_text'], rng, SYLL_SET, V2_PARAMS['p_empirical'],
                          V2_PARAMS['keyboard_strict'], V2_PARAMS['telex_share'], V2_PARAMS['telex_enabled'])
    if g is None:
        n_skipped_no_candidate += 1
        continue
    changed = [i for i, (a, b) in enumerate(zip(g['clean_text'].split(), g['noisy_text'].split())) if a != b]
    assert changed == [e['position'] for e in g['edits']], g['clean_text'][:60]
    if g['noisy_text'] != g['clean_text']:
        noisy_aug.append({'text': g['noisy_text'], 'corrected_text': g['clean_text']})
    if len(noisy_aug) >= n_noise:
        break

# Guard leakage: mọi sample augmentation phải nguồn từ corrected_text TRAIN (assert cứng)
_train_texts = {' '.join(canon_tokenize(r['corrected_text'])) for r in train_records}
for a in clean_aug + noisy_aug:
    assert ' '.join(canon_tokenize(a['corrected_text'])) in _train_texts, \
        f"Leakage: augmentation không nguồn từ train! ({a['corrected_text'][:60]!r})"
    assert unicodedata.is_normalized('NFC', a['text']) and unicodedata.is_normalized('NFC', a['corrected_text'])

train_data_run3 = [{'text': r['text'], 'corrected_text': r['corrected_text']} for r in train_records] \
    + clean_aug + noisy_aug
print('=== TẬP DỮ LIỆU HUẤN LUYỆN RUN 3 ===')
print(f'Thuần: {len(train_records)} · +sạch {len(clean_aug)} · +nhiễu v2 {len(noisy_aug)} '
      f'(skip {n_skipped_no_candidate} câu không có vị trí sinh được) · tổng {len(train_data_run3)} câu')
print('Noise v2 params:', V2_PARAMS)

=== TẬP DỮ LIỆU HUẤN LUYỆN RUN 3 ===
Thuần: 8343 · +sạch 2502 · +nhiễu v2 2502 (skip 0 câu không có vị trí sinh được) · tổng 13347 câu
Noise v2 params: {'p_empirical': 0.8, 'keyboard_strict': True, 'telex_share': 0.0, 'telex_enabled': True}


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class SpellingDataset(torch.utils.data.Dataset):  # copy nguyên nb3 §3
    def __init__(self, data_list, tokenizer, max_src_len=MAX_SOURCE_LEN, max_tgt_len=MAX_TARGET_LEN):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        src_enc = self.tokenizer(item['text'], max_length=self.max_src_len,
                                 truncation=True, padding=False)
        tgt_enc = self.tokenizer(item['corrected_text'], max_length=self.max_tgt_len,
                                 truncation=True, padding=False)
        return {'input_ids': src_enc['input_ids'],
                'attention_mask': src_enc['attention_mask'],
                'labels': tgt_enc['input_ids']}


val_dataset = SpellingDataset(val_records, tokenizer)
ds_run3 = SpellingDataset(train_data_run3, tokenizer)
print(f'Dataset sẵn sàng: Train R3={len(ds_run3)}, Val={len(val_dataset)}')

config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

Dataset sẵn sàng: Train R3=13347, Val=927


In [11]:
def build_lora_bartpho():  # copy nguyên nb3 §4
    print(f'Nạp mô hình cơ sở {MODEL_NAME} (torch_dtype=float16)...')
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if FP16 else torch.float32
    )
    model.gradient_checkpointing_enable()
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none"
    )
    lora_model = get_peft_model(model, peft_config)
    trainable_params, all_param = lora_model.get_nb_trainable_parameters()
    print(f'Trainable params: {trainable_params:,} / {all_param:,} ({100 * trainable_params / all_param:.2f}%)')
    return lora_model


collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors='pt')


def get_training_args(output_subdir):  # copy nguyên nb3 §5
    return Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR / output_subdir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.05,
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        fp16=FP16,
        predict_with_generate=False,
        report_to='none',
        load_best_model_at_end=True,
        metric_for_best_model='loss'
    )


print('=== BẮT ĐẦU HUẤN LUYỆN RUN 3 (AUGMENTATION + NOISE V2) ===')
model_run3 = build_lora_bartpho()
args_run3 = get_training_args('checkpoints_run3_noise_v2')
trainer_run3 = Seq2SeqTrainer(model=model_run3, args=args_run3, train_dataset=ds_run3,
                              eval_dataset=val_dataset, processing_class=tokenizer, data_collator=collator)
trainer_run3.train()

adapter_run3_dir = OUTPUT_DIR / 'lora_adapter_run3'
model_run3.save_pretrained(str(adapter_run3_dir))
print(f'Đã lưu adapter Run 3 vào {adapter_run3_dir}')

`torch_dtype` is deprecated! Use `dtype` instead!


=== BẮT ĐẦU HUẤN LUYỆN RUN 3 (AUGMENTATION + NOISE V2) ===
Nạp mô hình cơ sở vinai/bartpho-syllable (torch_dtype=float16)...


pytorch_model.bin:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

Trainable params: 4,718,592 / 482,514,944 (0.98%)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.269241,0.295654
2,0.942981,0.232544
3,0.781243,0.176270
4,0.736853,0.171875
5,0.685186,0.171021
6,0.627057,0.169434


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Đã lưu adapter Run 3 vào /kaggle/working/lora_adapter_run3


## 6. Inference + eval 3 chỉ số val/test — so trực diện Run 2

`batch_generate` greedy copy nb3 §7. So sánh Run 2: nếu có file `predictions_{val,test}_run2.jsonl` (optional input nb3) → tính lại **cùng pipeline** trong notebook này; không có → in số tham chiếu REPORT.md (nguồn ghi rõ). Kèm stratified non-word/real-word Run 3.

In [12]:
def batch_generate(model, tokenizer, texts, batch_size=16, beam_size=EVAL_BEAM_SIZE):  # copy nb3 §7
    model.eval()
    device = next(model.parameters()).device
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True,
                        max_length=MAX_SOURCE_LEN, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_length=MAX_TARGET_LEN,
                num_beams=beam_size,
                early_stopping=True if beam_size > 1 else False
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend([nfc_normalize(d) for d in decoded])
        if (i // batch_size) % 50 == 0:
            print(f'  Generated {min(i + batch_size, len(texts))}/{len(texts)} sentences...')
    return preds


val_texts = [r['text'] for r in val_records]
test_texts = [r['text'] for r in test_records]

print('--- INFERENCE RUN 3 (NOISE V2) · val ---')
preds_val_r3 = batch_generate(model_run3, tokenizer, val_texts)
print('--- INFERENCE RUN 3 (NOISE V2) · test 6k ---')
preds_test_r3 = batch_generate(model_run3, tokenizer, test_texts)

eval_val_r3 = evaluate_predictions(val_records, preds_val_r3, SYLL_SET)
eval_test_r3 = evaluate_predictions(test_records, preds_test_r3, SYLL_SET)

# So trực diện Run 2: recompute từ file nb3 nếu có; không có → tham chiếu REPORT.md (nguồn ghi rõ)
RUN2_REF = {  # nguồn: eval_report.json nb3 (chạy 23/09, seed 42, greedy) — REPORT.md §5.2/§12
    'val_f1': 0.7467,
    'test_f1': 0.8045, 'test_precision': 0.8667, 'test_recall': 0.7507,
    'test_overcorr': 0.0188, 'test_clean_retention': 0.8591,
}
run2_comparison = {'val_run2': None, 'test_run2': None,
                   'source': 'tham chiếu REPORT.md (không có file nb3 trong input)'}
if 'predictions_val_run2.jsonl' in OPTIONAL_PRESENT and 'predictions_test_run2.jsonl' in OPTIONAL_PRESENT:
    val2 = load_jsonl(OPTIONAL_PRESENT['predictions_val_run2.jsonl'])
    test2 = load_jsonl(OPTIONAL_PRESENT['predictions_test_run2.jsonl'])
    run2_comparison['val_run2'] = evaluate_predictions(val2, [r['prediction'] for r in val2], SYLL_SET)
    run2_comparison['test_run2'] = evaluate_predictions(test2, [r['prediction'] for r in test2], SYLL_SET)
    run2_comparison['source'] = 'recompute từ predictions nb3 (cùng pipeline trong notebook này)'


def _row(label, m):
    return (f"  {label:<14s} | {m['detection']['precision']:>7.2%} | {m['detection']['recall']:>7.2%} | "
            f"{m['detection']['f1']:>7.2%} | {m['correction']['accuracy']:>9.2%} | "
            f"{m['over_correction']['rate']:>9.2%} | {m['over_correction']['clean_retention_rate']:>9.2%}")


print('=' * 86)
print(f"  {'Set':<14s} | {'Prec':>7s} | {'Rec':>7s} | {'F1':>7s} | {'CorrAcc':>9s} | {'Over-corr':>9s} | {'CleanRet':>9s}")
print('-' * 86)
print(_row('VAL Run 3', eval_val_r3))
print(_row('TEST Run 3', eval_test_r3))
if run2_comparison['test_run2'] is not None:
    print(_row('VAL Run 2*', run2_comparison['val_run2']))
    print(_row('TEST Run 2*', run2_comparison['test_run2']))
    print('  (*: tính lại từ predictions nb3 — so sánh cùng pipeline)')
else:
    print(f"  [Tham chiếu REPORT.md · Run 2] VAL F1={RUN2_REF['val_f1']:.2%} · TEST F1={RUN2_REF['test_f1']:.2%} "
          f"Prec={RUN2_REF['test_precision']:.2%} Rec={RUN2_REF['test_recall']:.2%} "
          f"over-corr={RUN2_REF['test_overcorr']:.2%} clean-ret={RUN2_REF['test_clean_retention']:.2%}")
print('=' * 86)

print('\n=== STRATIFIED Run 3 (non-word vs real-word · proxy bảng âm tiết) ===')
for label, m in [('val', eval_val_r3), ('test', eval_test_r3)]:
    for cat in ('nonword', 'realword'):
        st = m['stratified'][cat]
        det = st['detected'] / st['gold'] if st['gold'] else 0.0
        corr = st['corrected'] / st['gold'] if st['gold'] else 0.0
        print(f'  {label:<5s} {cat:<9s} gold={st["gold"]:<6d} recall(detect)={det:.2%} corrected(gold)={corr:.2%}')
print('\nLưu ý: mục tiêu non-word [23,25]% chốt TRƯỚC khi chạy — stratified test chỉ report, không tối theo test.')

--- INFERENCE RUN 3 (NOISE V2) · val ---
  Generated 16/927 sentences...
  Generated 816/927 sentences...
--- INFERENCE RUN 3 (NOISE V2) · test 6k ---
  Generated 16/5983 sentences...
  Generated 816/5983 sentences...
  Generated 1616/5983 sentences...
  Generated 2416/5983 sentences...
  Generated 3216/5983 sentences...
  Generated 4016/5983 sentences...
  Generated 4816/5983 sentences...
  Generated 5616/5983 sentences...
  Set            |    Prec |     Rec |      F1 |   CorrAcc | Over-corr |  CleanRet
--------------------------------------------------------------------------------------
  VAL Run 3      |  76.06% |  72.53% |  74.25% |    86.73% |     0.98% |    66.67%
  TEST Run 3     |  85.68% |  72.75% |  78.68% |    67.38% |     1.98% |    86.52%
  VAL Run 2*     |  79.00% |  70.78% |  74.67% |    86.53% |     0.81% |    33.33%
  TEST Run 2*    |  86.67% |  75.07% |  80.45% |    67.94% |     1.88% |    85.91%
  (*: tính lại từ predictions nb3 — so sánh cùng pipeline)

=== STRATI

## 7. Post-processing — re-tune cùng rule-shape nb6 trên val Run 3

Rule engine R1–R4 copy từ nb6 (mining + veto + rebuild). **Mining lại FP trên val Run 3** (mỗi model có pattern FP riêng — không tái dùng whitelist/blacklist Run 2), sweep cùng grid G1–G6 lồng nhau, cùng tiêu chí pre-registered (gain ≥ 0,3 F1 val + over-correction không tăng) → frozen lên test Run 3. `postprocess_config.json` của nb6 chỉ nạp để **đối chiếu**, không áp lại.

In [13]:
def load_prediction_set(path):
    recs = load_jsonl(path)
    preds = [r['prediction'] for r in recs]
    assert len(recs) == len(preds)
    for r in recs:
        assert 'text' in r and 'corrected_text' in r and 'prediction' in r, r.get('row_id')
    return recs, preds


def mine_fp_blocks(records, preds):
    """Đếm FP theo edit block trên 1 prediction set (gold-aware — CHỈ dùng cho val, chống leakage DESIGN.md §9).
    Pure-FP block = block của pred mà không vị trí nguồn nào nằm trong gold positions.
    Block mixed (một phần vị trí là TP) không thể hoàn nguyên nguyên khối → chỉ thống kê.
    Trả về: deleted_tokens (block thuần xóa), fp_phrases (block >=2 token), punct_tokens (block punct-only), stats."""
    deleted_tokens = collections.Counter()
    fp_phrases = collections.Counter()
    punct_tokens = collections.Counter()
    stats = collections.Counter()
    for rec, pred in zip(records, preds):
        src = canon_tokenize(rec['text'])
        gold = canon_tokenize(rec['corrected_text'])
        ptoks = canon_tokenize(pred)
        gold_blocks = extract_edit_blocks(src, gold, levenshtein_opcodes(src, gold))
        gold_pos = {p for b in gold_blocks if b['src_span'][0] < b['src_span'][1]
                    for p in range(b['src_span'][0], b['src_span'][1])}
        pred_blocks = extract_edit_blocks(src, ptoks, levenshtein_opcodes(src, ptoks))
        for b in pred_blocks:
            if b['src_span'][0] >= b['src_span'][1]:
                continue  # insert block: không có vị trí nguồn để đối chiếu
            positions = set(range(b['src_span'][0], b['src_span'][1]))
            if positions & gold_pos:
                stats['mixed_blocks_not_revertible'] += 1
                continue
            stats['fp_blocks'] += 1
            stats['fp_type_' + b['type']] += 1
            if b['punct_only']:
                stats['fp_punct_only'] += 1
                for t in b['src_tokens']:
                    punct_tokens[t] += 1
            if b['type'] == 'delete':
                for t in b['src_tokens']:
                    deleted_tokens[nfc_normalize(t).lower()] += 1
            if len(b['src_tokens']) >= 2:
                fp_phrases[' '.join(nfc_normalize(t).lower() for t in b['src_tokens'])] += 1
    return {'deleted_tokens': deleted_tokens, 'fp_phrases': fp_phrases,
            'punct_tokens': punct_tokens, 'stats': stats}


def build_rule_resources(mining, k, phrase_min_count=PHRASE_MIN_COUNT):
    """Resource R3/R4 derive từ mining val (nguồn duy nhất của rule — không derive từ test)."""
    return {
        'r3_whitelist': {t for t, n in mining['deleted_tokens'].items() if n >= k} if k > 0 else set(),
        'r4_phrases': {p for p, n in mining['fp_phrases'].items() if n >= phrase_min_count},
    }


def veto_reason(block, cfg, res, syll_set):
    """Trả về tên rule nếu block bị phủ quyết (gold-agnostic khi áp: chỉ nhìn block shape + resource từ val)."""
    if block['src_span'][0] >= block['src_span'][1]:
        return None  # insert block: không hoàn nguyên được
    if cfg.get('r1_punct_guard') and block['punct_only']:
        return 'R1'
    is_pure_delete = block['tgt_span'][0] == block['tgt_span'][1]
    if is_pure_delete:
        low_toks = [nfc_normalize(t).lower() for t in block['src_tokens']]
        if cfg.get('r2_veto_valid_syllable') and low_toks and all(t in syll_set for t in low_toks):
            return 'R2'
        if cfg.get('r3_whitelist_min_count', 0) > 0 and low_toks \
                and all(t in res['r3_whitelist'] for t in low_toks):
            return 'R3'
    if cfg.get('r4_phrase_blacklist') and len(block['src_tokens']) >= 2:
        phrase = ' '.join(nfc_normalize(t).lower() for t in block['src_tokens'])
        if phrase in res['r4_phrases']:
            return 'R4'
    return None


def postprocess_one(text, pred, cfg, res, syll_set):
    """Áp rule lên 1 cặp (text, pred): veto block → hoàn nguyên tgt = src, rebuild bằng apply_edit_blocks.
    Không block nào bị veto → trả về prediction gốc NGUYÊN VẸN (identity guarantee)."""
    src = canon_tokenize(text)
    ptoks = canon_tokenize(pred)
    blocks = extract_edit_blocks(src, ptoks, levenshtein_opcodes(src, ptoks))
    hits = collections.Counter()
    changed = False
    for b in blocks:
        reason = veto_reason(b, cfg, res, syll_set)
        if reason:
            b['tgt_tokens'] = list(b['src_tokens'])
            b['tgt_span'] = list(b['src_span'])
            hits[reason] += 1
            changed = True
    if not changed:
        return pred, hits
    out_toks = apply_edit_blocks(src, blocks)
    out = ' '.join(out_toks)
    assert canon_tokenize(out) == out_toks, pred          # roundtrip token hóa
    assert unicodedata.is_normalized('NFC', out), pred    # NFC invariant
    return out, hits


def postprocess_preds(records, preds, cfg, res, syll_set):
    outs, hit_counts = [], collections.Counter()
    n_changed = 0
    for rec, pred in zip(records, preds):
        out, hits = postprocess_one(rec['text'], pred, cfg, res, syll_set)
        outs.append(out)
        if out != pred:
            n_changed += 1
        hit_counts.update(hits)
    return outs, {'n_changed_sentences': n_changed, 'veto_by_rule': dict(hit_counts)}


# --- Sanity case nhân tạo (không dùng dữ liệu thật) ---
_syll = {'hòa', 'học', 'sinh', 'an', 'tòa'}
_res1 = {'r3_whitelist': {'xyzq'}, 'r4_phrases': {'biểm họa'}}
_cfg_r12 = {'r1_punct_guard': True, 'r2_veto_valid_syllable': True,
            'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}

# identity prediction → nguyên vẹn
_t, _h = postprocess_one('học sinh hòa an', 'học sinh hòa an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh hòa an' and not _h
# R2: xóa 'hòa' (âm tiết hợp lệ) → hoàn tác
_t, _h = postprocess_one('học sinh hòa an', 'học sinh an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh hòa an' and _h['R2'] == 1, (_t, _h)
# R2 off → giữ nguyên xóa
_cfg_no = dict(_cfg_r12, r2_veto_valid_syllable=False)
_t, _h = postprocess_one('học sinh hòa an', 'học sinh an', _cfg_no, _res1, _syll)
assert _t == 'học sinh an' and not _h
# xóa non-word ngoài whitelist → R2 KHÔNG bắn (không hoàn tác oan), giữ xóa
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r12, _res1, _syll)
assert _t == 'học an' and not _h
# R3: token trong whitelist (k=1) → hoàn tác
_cfg_r13 = dict(_cfg_r12, r3_whitelist_min_count=1)
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r13, _res1, _syll)
assert _t == 'học xyzq an' and _h['R3'] == 1, (_t, _h)
# grid k=0 (không whitelist) → giữ xóa
_cfg_r2only = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
               'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r2only, _res1, _syll)
assert _t == 'học an' and not _h
# R1: xóa dấu câu → hoàn tác
_t, _h = postprocess_one('học sinh , an', 'học sinh an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh , an' and _h['R1'] == 1, (_t, _h)
# R4: phrase blacklist 2-token → hoàn tác cả block
_cfg_r4 = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
           'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': True}
_res4 = {'r3_whitelist': set(), 'r4_phrases': {'biểm họa'}}
_t, _h = postprocess_one('biểm họa lan', 'lan', _cfg_r4, _res4, _syll)
assert _t == 'biểm họa lan' and _h['R4'] == 1, (_t, _h)
print('Sanity rule engine PASS (identity · R1 · R2 on/off · R2 miss non-word · R3 whitelist · R4 phrase)')

Sanity rule engine PASS (identity · R1 · R2 on/off · R2 miss non-word · R3 whitelist · R4 phrase)


In [14]:
_pp_cfg_ref = json.loads(Path(INPUT_FILES['postprocess_config.json']).read_text(encoding='utf-8'))
print('Đối chiếu nb6 (Run 2):', _pp_cfg_ref['chosen_config_name'], _pp_cfg_ref['chosen_config'])
print('(Không áp lại config Run 2 — re-tune cùng rule-shape trên val Run 3, chống leakage DESIGN.md §9)')

mining_val_r3 = mine_fp_blocks(val_records, preds_val_r3)
print('== Mining FP · VAL Run 3 ==')
print('  stats:', dict(mining_val_r3['stats']))
print('  Top-10 token bị xóa oan:', mining_val_r3['deleted_tokens'].most_common(10))
print('  Top-5 phrase FP >=2 token:', mining_val_r3['fp_phrases'].most_common(5))

raw_val_r3_f1 = eval_val_r3['detection']['f1']
raw_val_r3_oc = eval_val_r3['over_correction']['rate']

GRID_R3 = [
    {'name': 'G1-r1',           'r1_punct_guard': True,  'r2_veto_valid_syllable': False, 'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False},
    {'name': 'G2-r1r2',         'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False},
    {'name': 'G3-r1r2-r3k1',    'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 1, 'r4_phrase_blacklist': False},
    {'name': 'G4-r1r2-r3k3',    'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 3, 'r4_phrase_blacklist': False},
    {'name': 'G5-r1r2-r3k5',    'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 5, 'r4_phrase_blacklist': False},
    {'name': 'G6-r1r2-r3k5-r4', 'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 5, 'r4_phrase_blacklist': True},
]
assert 1 <= len(GRID_R3) <= 6, 'Grid phải <=6 tổ hợp (chống overfit val n=927)'

sweep_rows_r3 = []
for cfg in GRID_R3:
    res = build_rule_resources(mining_val_r3, cfg['r3_whitelist_min_count'])
    post, post_stats = postprocess_preds(val_records, preds_val_r3, cfg, res, SYLL_SET)
    m = evaluate_predictions(val_records, post, SYLL_SET)
    sweep_rows_r3.append({
        'config': cfg['name'], 'cfg': cfg,
        'f1': m['detection']['f1'], 'precision': m['detection']['precision'],
        'recall': m['detection']['recall'], 'overcorr': m['over_correction']['rate'],
        'gain_f1': m['detection']['f1'] - raw_val_r3_f1,
        'n_changed': post_stats['n_changed_sentences'],
        'veto_by_rule': post_stats['veto_by_rule'],
    })

print(f"\nBaseline val Run 3 (raw): F1={raw_val_r3_f1:.2%} · over-corr={raw_val_r3_oc:.2%}")
print(f"  {'config':<18s} | {'dF1':>8s} | {'F1':>7s} | {'Prec':>7s} | {'Rec':>7s} | {'Over-corr':>9s} | {'#cau doi':>8s} | hit rule")
for row in sweep_rows_r3:
    print(f"  {row['config']:<18s} | {row['gain_f1']:>+8.2%} | {row['f1']:>7.2%} | {row['precision']:>7.2%} "
          f"| {row['recall']:>7.2%} | {row['overcorr']:>9.2%} | {row['n_changed']:>8d} | {row['veto_by_rule']}")

eligible_r3 = [r for r in sweep_rows_r3
               if r['overcorr'] <= raw_val_r3_oc and r['gain_f1'] >= MIN_VAL_GAIN]
_GRID_IDX_R3 = {c['name']: i for i, c in enumerate(GRID_R3)}
if eligible_r3:
    best_r3 = max(eligible_r3, key=lambda r: (r['f1'], -_GRID_IDX_R3[r['config']]))
    assert best_r3['gain_f1'] >= MIN_VAL_GAIN
    CHOSEN_CFG_R3 = dict(best_r3['cfg'])
    del CHOSEN_CFG_R3['name']
    CHOSEN_NAME_R3 = best_r3['config']
    print(f">> CHON: {CHOSEN_NAME_R3} · gain dF1 val Run 3 = {best_r3['gain_f1']:+.2%}")
else:
    CHOSEN_CFG_R3 = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
                     'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}
    CHOSEN_NAME_R3 = 'EMPTY-khong-vuot-nguong'
    print(f">> Khong config nao dat nguong (gain >= {MIN_VAL_GAIN:.1%} F1, over-corr khong tang) "
          '— Run 3 KHONG ap post-processing.')

CHOSEN_APPLIED_R3 = any([CHOSEN_CFG_R3['r1_punct_guard'], CHOSEN_CFG_R3['r2_veto_valid_syllable'],
                         CHOSEN_CFG_R3['r3_whitelist_min_count'] > 0, CHOSEN_CFG_R3['r4_phrase_blacklist']])
CHOSEN_RES_R3 = build_rule_resources(mining_val_r3, CHOSEN_CFG_R3['r3_whitelist_min_count'])

# Frozen áp lên val (báo cáo) + test Run 3
frozen_results_r3 = {}
POST_RUN3 = {}
print(f"\nFrozen config Run 3: {CHOSEN_NAME_R3} = {CHOSEN_CFG_R3} (applied={CHOSEN_APPLIED_R3})")
print(f"  {'set':<10s} | {'dPrec':>8s} | {'dRec':>8s} | {'dF1':>8s} | {'dOver-corr':>10s} | {'dCleanRet':>9s} | {'#doi':>5s} | hit rule")
for name, recs_, raw_preds in [('val_run3', val_records, preds_val_r3),
                               ('test_run3', test_records, preds_test_r3)]:
    raw = evaluate_predictions(recs_, raw_preds, SYLL_SET)
    if CHOSEN_APPLIED_R3:
        post, post_stats = postprocess_preds(recs_, raw_preds, CHOSEN_CFG_R3, CHOSEN_RES_R3, SYLL_SET)
    else:
        post, post_stats = list(raw_preds), {'n_changed_sentences': 0, 'veto_by_rule': {}}
    m = evaluate_predictions(recs_, post, SYLL_SET)
    frozen_results_r3[name] = {
        'raw': raw, 'post': m,
        'delta': {
            'precision': m['detection']['precision'] - raw['detection']['precision'],
            'recall': m['detection']['recall'] - raw['detection']['recall'],
            'f1': m['detection']['f1'] - raw['detection']['f1'],
            'overcorr': m['over_correction']['rate'] - raw['over_correction']['rate'],
            'clean_retention': m['over_correction']['clean_retention_rate'] - raw['over_correction']['clean_retention_rate'],
        },
        'postproc': post_stats,
    }
    POST_RUN3[name] = {'preds': post, 'stats': post_stats}
    d = frozen_results_r3[name]['delta']
    print(f"  {name:<10s} | {d['precision']:>+8.2%} | {d['recall']:>+8.2%} | {d['f1']:>+8.2%} "
          f"| {d['overcorr']:>+10.2%} | {d['clean_retention']:>+9.2%} | {post_stats['n_changed_sentences']:>5d} "
          f"| {post_stats['veto_by_rule']}")

_t3 = frozen_results_r3['test_run3']
print(f"\nRaw vs post-processed test Run 3: F1 {eval_test_r3['detection']['f1']:.2%} → {_t3['post']['detection']['f1']:.2%} · "
      f"over-correction {eval_test_r3['over_correction']['rate']:.2%} → {_t3['post']['over_correction']['rate']:.2%}")

Đối chiếu nb6 (Run 2): G3-r1r2-r3k1 {'r1_punct_guard': True, 'r2_veto_valid_syllable': True, 'r3_whitelist_min_count': 1, 'r4_phrase_blacklist': False}
(Không áp lại config Run 2 — re-tune cùng rule-shape trên val Run 3, chống leakage DESIGN.md §9)
== Mining FP · VAL Run 3 ==
  stats: {'mixed_blocks_not_revertible': 801, 'fp_blocks': 149, 'fp_type_substitute': 30, 'fp_type_delete': 111, 'fp_punct_only': 19, 'fp_type_multi': 2, 'fp_type_merge': 5, 'fp_type_split': 1}
  Top-10 token bị xóa oan: [(',', 11), ('hòa', 10), ('thủy', 9), ('khỏe', 8), ('và', 8), ('thỏa', 7), ('tòa', 7), ('tùy', 5), ('của', 4), ('các', 4)]
  Top-5 phrase FP >=2 token: [('và làm việc', 1), ('trong kĩ năng ôn tập', 1), ('bị tổn thất', 1), ('của đề tài', 1), (', giám sát', 1)]

Baseline val Run 3 (raw): F1=74.25% · over-corr=0.98%
  config             |      dF1 |      F1 |    Prec |     Rec | Over-corr | #cau doi | hit rule
  G1-r1              |   +0.58% |  74.83% |  77.38% |  72.44% |     0.91% |       19 | {'R1

## 8. Xuất file

`predictions_{val,test}_run3.jsonl` (schema nb3) · `eval_report_run3.json` (config + gate + calibration + 3 chỉ số + post-processing summary) · `postprocess_run3_report.json` (mining/sweep/frozen chi tiết) · `noise_model_v2.json` (đã ghi ở §4) · `lora_adapter_run3/`.

In [15]:
def save_predictions_jsonl(records, preds, out_path):  # copy nb3 §9
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p in zip(records, preds):
            f.write(json.dumps({'row_id': r.get('row_id'), 'text': r['text'],
                                'corrected_text': r['corrected_text'], 'prediction': p},
                               ensure_ascii=False) + '\n')


save_predictions_jsonl(val_records, preds_val_r3, OUTPUT_DIR / 'predictions_val_run3.jsonl')
save_predictions_jsonl(test_records, preds_test_r3, OUTPUT_DIR / 'predictions_test_run3.jsonl')

postprocess_run3_report = {
    'created': RUN_STAMP,
    'notebook': 'nb7_run3_train_eval',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'leakage_note': 'Re-tune cung rule-shape nb6 tren val Run 3; whitelist/blacklist derive MOI tu val Run 3.',
    'config_ref_run2': {'name': _pp_cfg_ref.get('chosen_config_name'),
                        'config': _pp_cfg_ref.get('chosen_config')},
    'mining_val_run3': {'stats': dict(mining_val_r3['stats']),
                        'top_deleted_tokens': mining_val_r3['deleted_tokens'].most_common(30),
                        'top_fp_phrases': mining_val_r3['fp_phrases'].most_common(15)},
    'baseline_val_run3': {'f1': raw_val_r3_f1, 'overcorr': raw_val_r3_oc},
    'sweep_val_run3': [{k: row[k] for k in ['config', 'f1', 'precision', 'recall', 'overcorr',
                                            'gain_f1', 'n_changed', 'veto_by_rule']}
                       for row in sweep_rows_r3],
    'selection': {'eligible': [r['config'] for r in eligible_r3], 'chosen': CHOSEN_NAME_R3,
                  'config': CHOSEN_CFG_R3, 'applied': CHOSEN_APPLIED_R3},
    'results_frozen': frozen_results_r3,
    'files': ['postprocess_run3_report.json'],
}
with open(OUTPUT_DIR / 'postprocess_run3_report.json', 'w', encoding='utf-8') as f:
    json.dump(postprocess_run3_report, f, ensure_ascii=False, indent=2)

eval_report_run3 = {
    'created': RUN_STAMP,
    'notebook': 'nb7_run3_train_eval',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'config': {
        'seed': SEED, 'model_name': MODEL_NAME,
        'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA, 'lora_dropout': LORA_DROPOUT,
        'target_modules': TARGET_MODULES, 'epochs': EPOCHS,
        'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM, 'learning_rate': LEARNING_RATE,
        'aug_clean_ratio': AUG_CLEAN_RATIO, 'aug_noise_ratio': AUG_NOISE_RATIO,
        'train_sentences': len(train_data_run3),
        'clean_aug': len(clean_aug), 'noisy_aug': len(noisy_aug),
        'noise_v2': V2_PARAMS,
    },
    'gate_telex': noise_model_v2['gate_telex'],
    'calibration': noise_model_v2['calibration'],
    'results': {
        'val_run3': eval_val_r3,
        'test_run3': eval_test_r3,
        'run2_comparison': {
            'source': run2_comparison['source'],
            'val_run2_f1': run2_comparison['val_run2']['detection']['f1'] if run2_comparison['val_run2'] else RUN2_REF['val_f1'],
            'test_run2_f1': run2_comparison['test_run2']['detection']['f1'] if run2_comparison['test_run2'] else RUN2_REF['test_f1'],
        },
    },
    'postprocessing': {
        'chosen': CHOSEN_NAME_R3, 'config': CHOSEN_CFG_R3, 'applied': CHOSEN_APPLIED_R3,
        'val_gain_f1': frozen_results_r3['val_run3']['delta']['f1'],
        'test_delta_f1': frozen_results_r3['test_run3']['delta']['f1'],
        'test_overcorr_delta': frozen_results_r3['test_run3']['delta']['overcorr'],
        'report_ref': 'postprocess_run3_report.json',
    },
    'files': ['predictions_val_run3.jsonl', 'predictions_test_run3.jsonl',
              'eval_report_run3.json', 'noise_model_v2.json', 'postprocess_run3_report.json',
              'lora_adapter_run3/'],
}
with open(OUTPUT_DIR / 'eval_report_run3.json', 'w', encoding='utf-8') as f:
    json.dump(eval_report_run3, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT nb7 (Run 3) ===')
print('Đã ghi vào', OUTPUT_DIR)
for fn in eval_report_run3['files']:
    print(' -', fn)

=== HOÀN TẤT nb7 (Run 3) ===
Đã ghi vào /kaggle/working
 - predictions_val_run3.jsonl
 - predictions_test_run3.jsonl
 - eval_report_run3.json
 - noise_model_v2.json
 - postprocess_run3_report.json
 - lora_adapter_run3/
